In [ ]:
import os
import warnings
import boto3
import joblib
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import roc_auc_score

#### Functions

In [ ]:
# plot optimization history
def plot_optimization_history(study, str_filename='output/optimization_history.png'):
    list_values = [trial.value for trial in study.trials if trial.value is not None]
    list_best = [max(list_values[:i+1]) for i in range(len(list_values))]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(range(len(list_values)), list_values, marker='o', markersize=4, alpha=0.5, color='steelblue', label='Trial AUC')
    ax.plot(range(len(list_best)), list_best, color='salmon', linewidth=2, label='Best AUC')
    ax.set_title('Optimization History', fontsize=16)
    ax.set_xlabel('Trial', fontsize=12)
    ax.set_ylabel('Validation AUC', fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

In [ ]:
# plot feature importance
def plot_feature_importance(model, list_feature_cols, str_filename='output/feature_importance.png'):
    arr_importance = model.feature_importances_
    df_importance = pd.DataFrame({
        'str_feature': list_feature_cols,
        'flt_importance': arr_importance,
    }).sort_values('flt_importance', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(5, len(list_feature_cols) * 0.4)))
    ax.barh(df_importance['str_feature'], df_importance['flt_importance'], color='steelblue', edgecolor='black')
    ax.set_title('Feature Importance (Gain)', fontsize=16)
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

#### Constants

In [ ]:
# bucket
str_bucket = os.getcwd().split('/')[4].replace('_', '-')
print(f'Bucket: {str_bucket}')

# step
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# s3 input path
str_s3_input = f's3://{str_bucket}/03_preprocessing'
print(f'S3 Input: {str_s3_input}')

# target
str_target = 'default_12m'
print(f'Target: {str_target}')

# columns to exclude from features
list_exclude_cols = [
    'loan_id',              # identifier
    'origination_date',     # used for splitting, not a feature
    'dob',                  # replaced by int_age
    'charged_off_amount',   # post-outcome leakage
    'paid_interest_amount', # post-outcome leakage
    str_target,             # target variable
]
print(f'Excluded columns: {list_exclude_cols}')

# number of optuna trials
int_n_trials = 50
print(f'Optuna trials: {int_n_trials}')

# output directory
os.makedirs('output', exist_ok=True)

#### Read Data

In [ ]:
# read preprocessed data
df_train = pd.read_parquet(f'{str_s3_input}/df_train_clean.parquet')
df_valid = pd.read_parquet(f'{str_s3_input}/df_valid_clean.parquet')
df_test = pd.read_parquet(f'{str_s3_input}/df_test_clean.parquet')

# print shapes
for str_name, df_split in [('Train', df_train), ('Validation', df_valid), ('Test', df_test)]:
    print(f'{str_name}: {df_split.shape}')

#### Define Features

Columns are excluded here rather than in preprocessing to keep the preprocessing pipeline fast and general-purpose. The following columns are excluded:
- **loan_id**: identifier with no predictive value
- **origination_date**: used for out-of-time splitting, not a model feature
- **dob**: raw date of birth, replaced by the engineered `int_age` feature
- **charged_off_amount, paid_interest_amount**: post-outcome variables that would cause data leakage (only known after loan resolution)
- **default_12m**: the target variable

In [ ]:
# define feature columns
list_feature_cols = [col for col in df_train.columns if col not in list_exclude_cols]
print(f'Features ({len(list_feature_cols)}): {list_feature_cols}')

# separate features and target
arr_X_train = df_train[list_feature_cols].values
arr_y_train = df_train[str_target].values

arr_X_valid = df_valid[list_feature_cols].values
arr_y_valid = df_valid[str_target].values

arr_X_test = df_test[list_feature_cols].values
arr_y_test = df_test[str_target].values

# calculate scale_pos_weight for class imbalance
flt_scale_pos_weight = (arr_y_train == 0).sum() / (arr_y_train == 1).sum()
print(f'Scale pos weight: {flt_scale_pos_weight:.2f}')

#### Bayesian Hyperparameter Tuning

Bayesian optimization (via Optuna) is used instead of grid or random search because it models the objective function and intelligently explores the hyperparameter space, converging on good configurations faster with fewer trials.

Early stopping is used during each trial to automatically determine the optimal number of boosting rounds. This prevents overfitting and eliminates `n_estimators` as a hyperparameter to tune.

The `scale_pos_weight` parameter is set to the ratio of negative to positive samples to account for class imbalance, which is common in credit risk (defaults are typically the minority class).

In [ ]:
# define optuna objective
def objective(trial):
    dict_params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'scale_pos_weight': flt_scale_pos_weight,
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'verbosity': 0,
    }

    model = xgb.XGBClassifier(
        **dict_params,
        n_estimators=1000,
        early_stopping_rounds=50,
    )

    model.fit(
        arr_X_train, arr_y_train,
        eval_set=[(arr_X_valid, arr_y_valid)],
        verbose=False,
    )

    arr_y_pred = model.predict_proba(arr_X_valid)[:, 1]
    flt_auc = roc_auc_score(arr_y_valid, arr_y_pred)
    return flt_auc

# run bayesian optimization
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=int_n_trials)
warnings.filterwarnings('default')

print(f'Best trial AUC: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

#### Optimization History

In [ ]:
plot_optimization_history(study)

#### Train Final Model

The final model is retrained using the best hyperparameters found by Optuna. Early stopping on the validation set is used again to determine the optimal number of boosting rounds for the final model.

In [ ]:
# train final model with best params
dict_best_params = study.best_params
dict_best_params['objective'] = 'binary:logistic'
dict_best_params['eval_metric'] = 'auc'
dict_best_params['scale_pos_weight'] = flt_scale_pos_weight
dict_best_params['verbosity'] = 0

model = xgb.XGBClassifier(
    **dict_best_params,
    n_estimators=1000,
    early_stopping_rounds=50,
)

model.fit(
    arr_X_train, arr_y_train,
    eval_set=[(arr_X_valid, arr_y_valid)],
    verbose=False,
)

# validation AUC
arr_y_pred_valid = model.predict_proba(arr_X_valid)[:, 1]
flt_auc_valid = roc_auc_score(arr_y_valid, arr_y_pred_valid)
print(f'Validation AUC: {flt_auc_valid:.4f}')
print(f'Best iteration: {model.best_iteration}')

#### Feature Importance

In [ ]:
plot_feature_importance(model, list_feature_cols)

#### Save

In [ ]:
# save model
joblib.dump(model, 'output/xgboost_model.joblib')
print('Model saved to output/xgboost_model.joblib')

# save best hyperparameters
df_params = pd.DataFrame([study.best_params])
df_params.to_csv('output/best_params.csv', index=False)
print('Best params saved to output/best_params.csv')

# save feature columns for reference
joblib.dump(list_feature_cols, 'output/feature_cols.joblib')
print('Feature columns saved to output/feature_cols.joblib')